# Lab 1: Chat completions

## Overview

Your team has one HTTPS endpoint for the hackathon. It speaks the OpenAI API, so the official `openai` Python package and most AI frameworks work without changes. Behind it, a 27-billion-parameter Qwen model runs on four GPUs in Akamai Cloud.

This lab walks through the chat completions endpoint: a first request, streaming, multi-turn conversations, output limits, long documents, thinking mode, and the quotas every team shares.

## Learning objectives

By the end of this lab you will be able to:

- Configure the `openai` client for the hackathon endpoint
- Send a chat request and read the response and its token usage
- Stream a response as it is generated
- Keep a multi-turn conversation going
- Control output length and know the hard limits
- Send a long document and ask questions about it
- Turn thinking mode on for one request and understand its cost
- Handle rate-limit responses without hammering the service

## Prerequisites

- Python 3.10 or newer
- Your team's API key from the organizers. Treat it like a password: never put it in frontend code or a public repository.
- A `.env` file in the repository root, copied from `.env.example` with your key filled in
- About 25 minutes

## Step 1: Set up your environment

The endpoint uses the same two settings the OpenAI SDK already understands: a base URL and an API key. Keeping them in `.env` means no key ever appears in a notebook.

> NOTE: If you already have an `OPENAI_API_KEY` for OpenAI in your shell, the cell below replaces it for this notebook only. Check the printed key starts with `hk_`.

In [ ]:
# Installs the packages this lab needs. Safe to rerun.
%pip install -q "openai>=3,<4" python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Reads OPENAI_BASE_URL and OPENAI_API_KEY from the .env file in the repository root.
# override=True makes the file win over any OPENAI_API_KEY already set in your shell.
load_dotenv(Path("..") / ".env", override=True)
load_dotenv(override=True)

base_url = os.environ["OPENAI_BASE_URL"]
api_key = os.environ["OPENAI_API_KEY"]
print("Endpoint:", base_url)
print("Key:     ", api_key[:6] + "..." + api_key[-4:])

client = OpenAI()  # The SDK reads the same two variables, so no arguments are needed.

## Step 2: Send your first request

A chat request is a list of messages. Each message has a `role` (`system`, `user`, or `assistant`) and `content`. The model name is the alias `hackathon-chat`; the server maps it to the real model.

Print the raw response first so you see the actual shape before pulling fields out of it.

In [ ]:
response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "In two sentences, what is a hackathon?"}],
)

# The whole object, exactly as the server returned it.
print(response.model_dump_json(indent=2))

In [ ]:
print("Answer:       ", response.choices[0].message.content)
print("Finish reason:", response.choices[0].finish_reason)
print("Tokens:       ", response.usage.prompt_tokens, "in,", response.usage.completion_tokens, "out")

The `model` field in the response shows the real model name. The `usage` block is what counts against your team's token quota, so it is worth printing while you develop.

## Step 3: Stream the response

The first token usually arrives in under a second, but a full answer takes several seconds. Streaming shows text as it is generated, which is what users expect from a chat interface.

With `stream=True` you receive chunks. Each chunk carries a small `delta`. The server also sends a final chunk with `usage`, so you still get the token count.

In [ ]:
stream = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Give me three names for a study-group app, one per line."}],
    stream=True,
    stream_options={"include_usage": True},
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
    if chunk.usage:  # Only the final chunk has this.
        print("\n\nTokens:", chunk.usage.completion_tokens, "out")

## Step 4: Hold a conversation

The server keeps no memory between requests. Every request stands alone. Watch what happens when a second request expects the model to remember the first.

In [ ]:
first = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "My name is Ada. Reply with just the word Noted."}],
)
print("Turn 1:", first.choices[0].message.content)

# A second request with no history. The server has already forgotten turn 1.
second = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "What is my name? Answer in one short sentence."}],
)
print("Turn 2 without history:", second.choices[0].message.content)

Your application owns the conversation. Keep a list of messages, append each reply, and send the whole list every turn. A `system` message at the top sets the assistant's behavior for the entire conversation.

In [ ]:
history = [
    {"role": "system", "content": "You are a concise assistant for a hackathon team."},
    {"role": "user", "content": "My name is Ada. Reply with just the word Noted."},
]
reply = client.chat.completions.create(model="hackathon-chat", messages=history)
history.append({"role": "assistant", "content": reply.choices[0].message.content})

history.append({"role": "user", "content": "What is my name? Answer in one short sentence."})
reply = client.chat.completions.create(model="hackathon-chat", messages=history)
print("Turn 2 with history:", reply.choices[0].message.content)

## Step 5: Control the output

Three limits apply to every request:

| Setting | Behavior |
|---|---|
| `max_tokens` not set | 512 output tokens |
| `max_tokens` above 4096 | Lowered to 4096 by the server |
| `n` | Always 1 |

When the cap cuts a reply short, `finish_reason` is `length` instead of `stop`. Check it when answers look truncated.

In [ ]:
short = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Explain how a hash map works."}],
    max_tokens=20,
)
print(short.choices[0].message.content)
print("\nFinish reason:", short.choices[0].finish_reason)

In [ ]:
big = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Reply with the single word OK."}],
    max_tokens=100000,  # Above the cap. The server lowers it to 4096 instead of rejecting the request.
)
print(big.choices[0].message.content, "| finish reason:", big.choices[0].finish_reason)

`temperature` controls randomness. The server applies the model's recommended defaults when you leave it out. Use `0` when you want the same answer every time, for example when parsing the output in code.

In [ ]:
for temperature in (0.0, 1.0):
    r = client.chat.completions.create(
        model="hackathon-chat",
        messages=[{"role": "user", "content": "Name one color. One word only."}],
        temperature=temperature,
        max_tokens=5,
    )
    print(f"temperature={temperature}: {r.choices[0].message.content.strip()}")

## Step 6: Work with long documents

The context window is 32,768 tokens, shared between your input and the model's output. In practice about 30,000 input tokens work, which is roughly 60 pages of prose. A token is about three quarters of an English word.

There is no file upload. Extract text from PDFs or spreadsheets in your own code, then put it in the prompt. The cell below builds a 15-page document with one fact buried in the middle.

In [ ]:
filler = ("The committee reviewed the quarterly logistics report and noted that shipping volumes in the "
          "northern region remained steady while maintenance costs rose modestly. Several members "
          "proposed revisiting the vendor contracts next spring. ")
pages = [filler * 12 for _ in range(15)]
pages[9] += " IMPORTANT: the warehouse door code is 7391-ORCHID. "
document = "\n\n".join(f"Page {i + 1}\n{page}" for i, page in enumerate(pages))
print("Words in document:", len(document.split()))

response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": document + "\n\nQuestion: what is the warehouse door code? Answer with just the code."}],
    max_tokens=30,
)
print("Answer:       ", response.choices[0].message.content.strip())
print("Prompt tokens:", response.usage.prompt_tokens)

> NOTE: Requests above the context window are rejected with an error, not truncated. If you build a chat app, trim old turns from the history before that happens.

## Step 7: Thinking mode

The model can reason step by step before answering. Thinking is **off by default** so a normal request returns an answer, not a page of reasoning. You can turn it on for a single request.

Two things to know first. The reasoning arrives inside `<think>` tags at the start of `content`, and it uses your `max_tokens` budget. Measured on this endpoint with 40 other users active, 2,048 thinking tokens took about 155 seconds. Budget both tokens and time.

In [ ]:
import time

start = time.time()
response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "A bat and a ball cost $1.10 together. The bat costs $1.00 more than the ball. What does the ball cost?"}],
    max_tokens=1500,
    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
)
text = response.choices[0].message.content
reasoning, _, answer = text.partition("</think>")

print("Reasoning (first 400 characters):", reasoning[:400].strip(), "...\n")
print("Answer:", answer.strip() or "(cut off before the answer; raise max_tokens)")
print(f"\nFinish reason: {response.choices[0].finish_reason} | {response.usage.completion_tokens} tokens in {time.time() - start:.0f}s")

If `finish_reason` is `length`, the budget ran out during the reasoning and there is no answer. Use thinking for a few hard questions, not for every request.

## Step 8: Respect the quotas

One key per team, shared by every member and every script you run. The limits are per team, per minute:

| Quota | Limit |
|---|---|
| Requests, all three endpoints combined | 60 per minute |
| Chat tokens, input plus output | 100,000 per minute |
| Image requests | 6 per minute |

When you exceed one, the server answers `429 Too Many Requests` with an `x-ratelimit-reset` header saying how many seconds until the window resets. The SDK retries twice on its own with a backoff. For anything that runs in a loop, handle it explicitly and add a little randomness so several scripts do not all retry at the same instant.

In [ ]:
import random
import openai

client = OpenAI(max_retries=0)  # Handle retries ourselves so the behavior is visible.

def ask(prompt, attempts=5):
    for attempt in range(attempts):
        try:
            r = client.chat.completions.create(
                model="hackathon-chat",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=60,
            )
            return r.choices[0].message.content
        except openai.RateLimitError as e:
            wait = int(e.response.headers.get("x-ratelimit-reset", 5)) + random.uniform(0, 2)
            print(f"Rate limited; waiting {wait:.0f}s before attempt {attempt + 2}")
            time.sleep(wait)
    raise RuntimeError("Gave up after repeated rate limits")

print(ask("One sentence of advice for a hackathon team."))

client = OpenAI()  # Back to the default client with built-in retries.

## Step 9: Know what is not there

The endpoint exposes three operations: `POST /v1/chat/completions`, `POST /v1/embeddings`, and `POST /v1/images/generations`. Everything else in the OpenAI API returns `404`. The cell below shows the most common surprise, listing models.

In [ ]:
try:
    client.models.list()
except openai.APIStatusError as e:
    print("GET /v1/models is not exposed. Status:", e.status_code)

Also not available: the Responses API, audio, image editing, image input to the chat model, and choosing other model names. Tool calling has not been tested on this deployment; if your project depends on it, try it early.

## Validation

Run the cell below. It confirms your key, the endpoint, and token accounting all work.

In [ ]:
check = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Reply with the single word READY."}],
    max_tokens=5,
)
assert "READY" in check.choices[0].message.content.upper(), check.choices[0].message.content
assert check.usage.total_tokens > 0
print("Lab 1 complete.")

## Clean up

This lab created no cloud resources. The only thing to protect is the `.env` file with your key. Delete it when the hackathon ends, and never commit it.

## Next steps

Lab 2 adds the embeddings endpoint and builds a small semantic search, then combines it with chat for retrieval-augmented answers.